# RSNA Knee — Inference (submission notebook)

Offline, <=9h. Re-executes against the hidden test set at scoring time.
Attached datasets supply everything: `knee-code` (the package), `knee-weights`
(per-plane checkpoints). Output must be `/kaggle/working/submission.csv`.

Ensemble: each checkpoint declares its series type; each study fans out to every
model (strict typing, missing plane = model sits out) and rows merge by NaN-aware
mean. `USE_CONSTANT_PRIORS = True` reproduces the E000 mechanics-validation run.

In [ ]:
# No internet: code arrives as an attached dataset, not pip.
import sys

sys.path.append("/kaggle/input/knee-code")

from knee.infer import predict_studies
from knee.labels import SUBMISSION_COLUMNS

USE_CONSTANT_PRIORS = False  # True = E000 mechanics check, no model in the loop

In [ ]:
from pathlib import Path

import pandas as pd

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug>; accept either — verified on the train kernel 2026-08-31.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP = next(p for p in candidates if (p / "test.csv").exists())
WEIGHTS = Path("/kaggle/input/knee-weights")
test = pd.read_csv(COMP / "test.csv")  # ~1300 studies at scoring time

In [ ]:
checkpoints = sorted(WEIGHTS.glob("*.pt"))
print([p.name for p in checkpoints])

In [ ]:
# The 9h budget lives here: each model decodes only its own series per study,
# and predict_studies never aborts on a bad study (a crash costs a submission).
if USE_CONSTANT_PRIORS:
    sub = pd.DataFrame(
        [[uid, *([0.5] * 12)] for uid in test["StudyInstanceUID"]],
        columns=list(SUBMISSION_COLUMNS),
    )
else:
    sub = predict_studies(COMP, checkpoints)

In [ ]:
assert list(sub.columns) == list(SUBMISSION_COLUMNS)
assert len(sub) == len(test) and not sub.isna().any().any()
sub.to_csv("submission.csv", index=False)  # exact filename required
sub.head()